In [5]:
import cv2
import numpy as np


def load_class_names(namesfile):
    class_names = []
    with open(namesfile, "r") as fp:
        for line in fp.readlines():
            class_names.append(line.rstrip())
    return class_names


video_path = "1.mp4"
cfg_path = "yolov3.cfg"
weights_path = "yolov3.weights"
names_path = "coco.names"

net = cv2.dnn.readNetFromDarknet(cfg_path, weights_path)
ln = net.getLayerNames()
ln = [ln[i - 1] for i in net.getUnconnectedOutLayers().flatten()]

class_names = load_class_names(names_path)
cap = cv2.VideoCapture(video_path)

frame_count = 0
max_objects = 0
max_classes = 0
frame11_best = None

while True:
    ret, img = cap.read()
    if not ret:
        break

    h, w = img.shape[:2]
    blob = cv2.dnn.blobFromImage(img, 1 / 255.0, (416, 416), swapRB=True, crop=False)
    net.setInput(blob)
    layerOutputs = net.forward(ln)

    boxes = []
    confidences = []
    classIDs = []

    for output in layerOutputs:
        for detection in output:
            scores = detection[5:]
            classID = int(np.argmax(scores))
            confidence = float(scores[classID])

            if confidence > 0.5:
                box = detection[0:4] * np.array([w, h, w, h])
                centerX, centerY, width, height = box.astype("int")
                x = int(centerX - width / 2)
                y = int(centerY - height / 2)
                boxes.append([x, y, int(width), int(height)])
                confidences.append(confidence)
                classIDs.append(classID)

    idxs = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.3)

    detections = []
    if len(idxs) > 0:
        idxs = idxs.flatten()
        for i in idxs:
            detections.append(
                {
                    "class_id": classIDs[i],
                    "label": class_names[classIDs[i]],
                    "confidence": confidences[i],
                    "x": boxes[i][0],
                    "y": boxes[i][1],
                    "w": boxes[i][2],
                    "h": boxes[i][3],
                }
            )

    max_objects = max(max_objects, len(detections))
    max_classes = max(max_classes, len(set(d["label"] for d in detections)))

    if frame_count == 11 and len(detections) > 0:
        frame11_best = max(detections, key=lambda d: d["confidence"])

    frame_count += 1

cap.release()

print(frame_count)
print(max_objects)
print(max_classes)

if frame11_best is not None:
    print(round(frame11_best["confidence"], 3))
    print(frame11_best["label"])
    print(int(frame11_best["x"]))
    print(int(frame11_best["y"]))
    print(int(frame11_best["w"]))
    print(int(frame11_best["h"]))
else:
    print("No detections on frame 11")


100
10
4
0.999
person
1091
295
164
460


In [4]:
import cv2
import numpy as np


def load_class_names(namesfile):
    class_names = []
    with open(namesfile, "r", encoding="utf-8") as fp:
        for line in fp.readlines():
            class_names.append(line.rstrip())
    return class_names


video_path = "1.mp4"
cfg_path = "yolov3.cfg"
weights_path = "yolov3.weights"
names_path = "coco.names"

net = cv2.dnn.readNetFromDarknet(cfg_path, weights_path)

layer_names = net.getLayerNames()
try:
    ln = [layer_names[i - 1] for i in net.getUnconnectedOutLayers().flatten()]
except:
    ln = [layer_names[i[0] - 1] for i in net.getUnconnectedOutLayers()]

class_names = load_class_names(names_path)

cap = cv2.VideoCapture(video_path)

frame_count = 0
max_objects_in_frame = 0
max_classes_in_frame = 0
frame11_best = None

while True:
    ret, img = cap.read()
    if not ret:
        break

    h, w = img.shape[:2]

    blob = cv2.dnn.blobFromImage(img, 1 / 255.0, (416, 416), swapRB=True, crop=False)
    net.setInput(blob)
    layerOutputs = net.forward(ln)

    boxes = []
    confidences = []
    classIDs = []

    for output in layerOutputs:
        for detection in output:
            scores = detection[5:]
            classID = int(np.argmax(scores))
            confidence = float(scores[classID])

            if confidence > 0.5:
                box = detection[0:4] * np.array([w, h, w, h])
                centerX, centerY, width, height = box.astype("int")

                x = int(centerX - width / 2)
                y = int(centerY - height / 2)

                boxes.append([x, y, int(width), int(height)])
                confidences.append(confidence)
                classIDs.append(classID)

    idxs = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.3)

    detections = []
    if len(idxs) > 0:
        try:
            idxs = idxs.flatten()
        except:
            idxs = np.array(idxs).flatten()

        for i in idxs:
            x, y, bw, bh = boxes[i]
            detections.append(
                {
                    "label": class_names[classIDs[i]],
                    "confidence": float(confidences[i]),
                    "x": int(x),
                    "y": int(y),
                    "w": int(bw),
                    "h": int(bh),
                }
            )

    max_objects_in_frame = max(max_objects_in_frame, len(detections))
    max_classes_in_frame = max(
        max_classes_in_frame, len(set(d["label"] for d in detections))
    )

    if frame_count == 11 and len(detections) > 0:
        frame11_best = max(detections, key=lambda d: d["confidence"])

    frame_count += 1

cap.release()

print(frame_count)
print(max_objects_in_frame)
print(max_classes_in_frame)

if frame11_best is not None:
    print(round(frame11_best["confidence"], 3))
    print(frame11_best["label"])
    print(frame11_best["x"])
    print(frame11_best["y"])
    print(frame11_best["w"])
    print(frame11_best["h"])
else:
    print("none")
    print("none")
    print("none")
    print("none")
    print("none")
    print("none")


100
10
4
0.999
person
1091
295
164
460
